# Cell-crop labeler

Drop raw images into `to_be_labeled/`, run the cells below, then label
by pressing one key per cell:

* `8` → sickle
* `9` → non_sickle
* `0` → ambiguous
* `z` → undo last label
* `s` → skip without labeling (advance only)

Labels are appended to `labels/labels.csv` immediately after each keypress.
Already-labeled `(source_image, x, y)` triples are skipped on the next run,
so you can stop and resume freely.

In [1]:
%load_ext autoreload
%autoreload 2

import csv
import os
import tkinter as tk
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageTk
from scipy import ndimage as ndi
from tqdm.auto import tqdm

from sickling.rbc_classification.py_modules.config import load_config
from sickling.rbc_classification.py_modules.io.images import find_raw_image, load_raw_greyscale, normalize_image
from sickling.rbc_classification.py_modules.stage1_unet import load_unet, predict_label_map
from sickling.rbc_classification.py_modules.stage2_instances.watershed import mask_to_instances_with_reasons

In [2]:
# NOTE: this sys.path patch is a fallback. The proper fix is to run
# `pip install -e .` once at the repo root, after which `sickling` imports
# from anywhere — including notebooks — without any boilerplate.
import sys
from pathlib import Path

project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
# ===========================================================================
# CONFIG
# ===========================================================================
ANNOTATOR_NAME = "UG"
MODEL_PATH = Path(project_root) / "models" / "unet_fold_2_best.pth"
TO_BE_LABELED = Path(project_root) / "to_be_labeled"
CSV_OUTPUT_PATH = Path(project_root) / "labels" / "labels.csv"
DISPLAY_HALF = 192               # window half-size used for the cell preview (→ 384x384 box)
DISPLAY_SCALE = 3                # nearest-neighbour upscale for the GUI
MAX_PER_IMAGE = None             # set to e.g. 50 for a quick pass per FOV

# --- Priority-sort knobs ---
PRIORITY_SORT_BY_POLYMER = False   # sort whole queue by polymer_score descending
MIN_POLYMER_SCORE = 0             # drop cells with score < this (0 = show all)
BOUNDARY_BAND_ITERATIONS = 4      # erosion depth that defines the boundary band

# --- Display outline ---
CIRCLE_RADIUS_FACTOR = 1.5

# --- Redo mode ---
# True = iterate already-labeled rows in LABELS_CSV_FOR_REDO. Each keypress
# overwrites the existing label. 'z' undo restores the previous label, 's' keeps
# the current label. REDO_CLASSES filters which existing labels you want to
# revisit — e.g. {"ambiguous"} to triage just the borderline calls.
REDO_MODE = False
LABELS_CSV_FOR_REDO = CSV_OUTPUT_PATH
REDO_CLASSES: set[str] = {"sickle", "non_sickle", "ambiguous"}

# --- Note hotkeys ---
# Pressing 1/2/3 inserts the corresponding canned note into the entry box.
# (You can also click into the entry to type free-form text.)
NOTE_SHORTCUTS: dict[str, str] = {
    "1": "nearby debris",
    "2": "exogenous polymer",
    "3": "multiple sickle",
}

KEY_SICKLE = "8"
KEY_NON_SICKLE = "9"
KEY_AMBIGUOUS = "0"
KEY_UNDO = "z"
KEY_SKIP = "s"

cfg = load_config()
print("to_be_labeled :", TO_BE_LABELED)
print("model         :", MODEL_PATH)
print("labels.csv    :", CSV_OUTPUT_PATH)
print("redo mode     :", REDO_MODE, " | redo classes:", REDO_CLASSES if REDO_MODE else "(n/a)")
print("min_area, max_area, peak_min_distance :",
      cfg.instances.min_area, cfg.instances.max_area, cfg.instances.peak_min_distance)
print("priority sort by polymer:", PRIORITY_SORT_BY_POLYMER,
      "| min polymer score:", MIN_POLYMER_SCORE)
print("note shortcuts:", NOTE_SHORTCUTS)

to_be_labeled : E:\utku g leica\rbc-class\to_be_labeled
model         : E:\utku g leica\rbc-class\models\unet_fold_2_best.pth
labels.csv    : E:\utku g leica\rbc-class\labels\labels.csv
redo mode     : False  | redo classes: (n/a)
min_area, max_area, peak_min_distance : 550 6000 12
priority sort by polymer: False | min polymer score: 0
note shortcuts: {'1': 'nearby debris', '2': 'exogenous polymer', '3': 'multiple sickle'}


In [4]:
def _read_existing_records(csv_path: Path) -> set[tuple[str, str, str]]:
    if not csv_path.exists():
        return set()
    out: set[tuple[str, str, str]] = set()
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row.get("source_image"):
                continue
            out.add((row["source_image"], str(row["x"]), str(row["y"])))
    return out


def _ensure_csv_header(csv_path: Path) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    if csv_path.exists() and csv_path.stat().st_size > 0:
        return
    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["source_image", "x", "y", "label", "annotator", "notes"])


def _polymer_score_at_boundary(
    label_map, instance_image, instance_id, polymer_class,
    iterations=BOUNDARY_BAND_ITERATIONS,
):
    this_mask = instance_image == instance_id
    if not this_mask.any():
        return 0
    eroded = ndi.binary_erosion(this_mask, iterations=iterations)
    boundary_band = this_mask & ~eroded
    return int(((label_map == polymer_class) & boundary_band).sum())


def _max_centroid_distance(instance_image, instance_id, cy, cx) -> float:
    rows, cols = np.where(instance_image == instance_id)
    if rows.size == 0:
        return 0.0
    return float(np.sqrt((rows - cy) ** 2 + (cols - cx) ** 2).max())


def _make_display_crop(raw_norm, cy, cx, display_radius, half=DISPLAY_HALF) -> Image.Image:
    h, w = raw_norm.shape
    pad = half
    raw_padded = np.pad(raw_norm, pad, mode="constant", constant_values=0.0)
    cy_p, cx_p = cy + pad, cx + pad
    raw_window = raw_padded[cy_p - half : cy_p + half, cx_p - half : cx_p + half]
    rgb = (np.stack([raw_window] * 3, axis=-1) * 255).clip(0, 255).astype(np.uint8)
    img = Image.fromarray(rgb)

    r = int(round(display_radius))
    if r > 0:
        ImageDraw.Draw(img).ellipse(
            (half - r, half - r, half + r, half + r),
            outline=(100, 50, 255), width=2,
        )

    arr = np.array(img)
    cy_c, cx_c = half, half
    arr[cy_c, max(cx_c - 4, 0) : min(cx_c + 5, 2 * half)] = [255, 255, 0]
    arr[max(cy_c - 4, 0) : min(cy_c + 5, 2 * half), cx_c] = [255, 255, 0]
    return Image.fromarray(arr)


def _match_instance_at(point_x, point_y, instance_image) -> int:
    h, w = instance_image.shape
    if 0 <= point_y < h and 0 <= point_x < w:
        return int(instance_image[point_y, point_x])
    return 0


def build_queue(
    seg_cache: dict[str, dict],
    cfg,
    existing_records: set[tuple[str, str, str]],
    max_per_image: int | None = None,
    priority_sort: bool = PRIORITY_SORT_BY_POLYMER,
    min_polymer_score: int = MIN_POLYMER_SCORE,
) -> list[dict]:
    """Walk every kept instance in every cached FOV and build the to-label queue.
    Skips anything already present in ``existing_records``."""
    queue: list[dict] = []
    for source_image, entry in seg_cache.items():
        raw_norm = entry["raw_norm"]
        label_map = entry["label_map"]
        instance_image = entry["instance_image"]
        n_kept = int(instance_image.max(initial=0))
        added = 0
        for iid in range(1, n_kept + 1):
            rows, cols = np.where(instance_image == iid)
            if rows.size == 0:
                continue
            cy = int(round(rows.mean()))
            cx = int(round(cols.mean()))
            if (source_image, str(cx), str(cy)) in existing_records:
                continue
            polymer_score = _polymer_score_at_boundary(
                label_map, instance_image, iid, cfg.classes.polymer
            )
            if polymer_score < min_polymer_score:
                continue
            display_radius = CIRCLE_RADIUS_FACTOR * _max_centroid_distance(
                instance_image, iid, cy, cx
            )
            queue.append({
                "source_image": source_image,
                "x": cx, "y": cy,
                "polymer_score": polymer_score,
                "display_radius": display_radius,
                "current_label": None,
                "current_notes": "",
                "display": _make_display_crop(raw_norm, cy, cx, display_radius),
            })
            added += 1
            if max_per_image is not None and added >= max_per_image:
                break

    if priority_sort:
        queue.sort(key=lambda r: r["polymer_score"], reverse=True)
    return queue


def build_redo_queue(
    labels_csv: Path,
    seg_cache: dict[str, dict],
    cfg,
    redo_classes: set[str] = None,
) -> list[dict]:
    """Iterate rows already in ``labels_csv`` whose ``label`` is in
    ``redo_classes``. Reads from the segmentation cache so the labeling-config
    cells stay cheap to re-run."""
    redo_classes = redo_classes or REDO_CLASSES
    df = pd.read_csv(labels_csv)
    if df.empty:
        return []
    queue: list[dict] = []
    for source_image, fov_rows in df.groupby("source_image"):
        entry = seg_cache.get(str(source_image))
        if entry is None:
            print(f"⚠ {source_image} not in segmentation cache — skipping {len(fov_rows)} row(s).")
            continue
        raw_norm = entry["raw_norm"]
        label_map = entry["label_map"]
        instance_image = entry["instance_image"]
        for _, row in fov_rows.iterrows():
            current_label = str(row["label"])
            if current_label not in redo_classes:
                continue
            cx, cy = int(row["x"]), int(row["y"])
            iid = _match_instance_at(cx, cy, instance_image)
            if iid > 0:
                display_radius = CIRCLE_RADIUS_FACTOR * _max_centroid_distance(
                    instance_image, iid, cy, cx
                )
                polymer_score = _polymer_score_at_boundary(
                    label_map, instance_image, iid, cfg.classes.polymer
                )
            else:
                display_radius = 40.0
                polymer_score = 0
            queue.append({
                "source_image": str(source_image),
                "x": cx, "y": cy,
                "polymer_score": polymer_score,
                "display_radius": display_radius,
                "current_label": current_label,
                "current_notes": str(row.get("notes", "") or "").strip(),
                "display": _make_display_crop(raw_norm, cy, cx, display_radius),
            })
    return queue


_ensure_csv_header(CSV_OUTPUT_PATH)
existing = _read_existing_records(CSV_OUTPUT_PATH)
print(f"Already labeled (from {CSV_OUTPUT_PATH.name}): {len(existing)}")


Already labeled (from labels.csv): 2591


In [5]:
model = load_unet(MODEL_PATH, n_classes=4)
print("U-Net loaded on", next(model.parameters()).device)

U-Net loaded on cuda:0


## Segmentation cache (run once per session)

This cell runs U-Net + watershed for every FOV that may show up in the queue
and caches `(raw_norm, label_map, instance_image, pre_instance_image, drop_reasons)`
in memory. Re-running the **labeling-config** cells below (e.g. flipping
`REDO_MODE`, tweaking `MIN_POLYMER_SCORE`, changing `REDO_CLASSES`) does NOT
re-segment — only re-running this cell does.


In [6]:
def _collect_fov_paths_for_session() -> list[Path]:
    """Union of FOVs that may be needed: every image in `to_be_labeled/` plus
    every source_image present in `LABELS_CSV_FOR_REDO` (when REDO_MODE)."""
    seen: dict[str, Path] = {}
    raw_images_fallback = Path(project_root) / "raw_images"

    for ext in ("jpg", "jpeg", "png", "tif", "tiff"):
        for p in sorted(TO_BE_LABELED.glob(f"*.{ext}")):
            seen.setdefault(p.name, p)

    if REDO_MODE and LABELS_CSV_FOR_REDO.exists():
        df = pd.read_csv(LABELS_CSV_FOR_REDO)
        for src in df["source_image"].dropna().unique():
            if src in seen:
                continue
            stem = Path(str(src)).stem
            p = find_raw_image(stem, TO_BE_LABELED) or find_raw_image(stem, raw_images_fallback)
            if p is not None:
                seen[p.name] = p
    return list(seen.values())


def prepare_segmentation_cache(fov_paths, model, cfg) -> dict[str, dict]:
    """Run U-Net + watershed for every FOV. Returns ``{filename: dict}``."""
    cache: dict[str, dict] = {}
    for raw_path in tqdm(fov_paths, desc="Segmenting FOVs"):
        raw = load_raw_greyscale(raw_path)
        raw_norm = normalize_image(raw, cfg.crop.norm_percentile)
        label_map = predict_label_map(model, raw_norm, n_classes=4)
        inst, _stats, pre, reasons = mask_to_instances_with_reasons(
            label_map, cfg.instances, cfg.classes
        )
        cache[raw_path.name] = {
            "raw_path": raw_path,
            "raw_norm": raw_norm,
            "label_map": label_map,
            "instance_image": inst,
            "pre_instance_image": pre,
            "drop_reasons": reasons,
        }
    return cache


# Run once per session (re-run only when MODEL_PATH or cfg.instances change).
_fov_paths = _collect_fov_paths_for_session()
print(f"Will segment {len(_fov_paths)} FOV(s).")
SEG_CACHE = prepare_segmentation_cache(_fov_paths, model, cfg)
print(f"SEG_CACHE has {len(SEG_CACHE)} entries.")


Will segment 355 FOV(s).


Segmenting FOVs:   0%|          | 0/355 [00:00<?, ?it/s]

SEG_CACHE has 355 entries.


## Build the unlabeled queue

For each raw image in `to_be_labeled/`:
1. Percentile-normalize → sliding-window U-Net → 4-class label map.
2. Watershed → kept instances (drops edge-touching, below `min_area`, above `max_area`).
3. For each kept instance: take its centroid as `(x, y)`, build a display crop with the cell outlined.
4. Skip any `(source_image, x, y)` already in `labels.csv`.

In [7]:
if REDO_MODE:
    queue = build_redo_queue(LABELS_CSV_FOR_REDO, SEG_CACHE, cfg, redo_classes=REDO_CLASSES)
    print(f"Redo mode: {len(queue)} labeled rows from {LABELS_CSV_FOR_REDO.name} "
          f"(filter classes: {REDO_CLASSES})")
else:
    queue = build_queue(SEG_CACHE, cfg, existing_records=existing, max_per_image=MAX_PER_IMAGE)
    print(f"To label this session: {len(queue)} crops")

if queue:
    print("first row :", {k: v for k, v in queue[0].items() if k != "display"})
    print("last row  :", {k: v for k, v in queue[-1].items() if k != "display"})
    scores = np.array([r["polymer_score"] for r in queue])
    print(
        f"polymer score: min={scores.min()} median={int(np.median(scores))} "
        f"p90={int(np.percentile(scores, 90))} max={scores.max()} "
        f"(n_with_score>0 = {int((scores > 0).sum())})"
    )


To label this session: 86248 crops
first row : {'source_image': 'D16_03_1_10_Bright Field_001.jpg', 'x': 1451, 'y': 1210, 'polymer_score': 0, 'display_radius': 74.48825410761081, 'current_label': None, 'current_notes': ''}
last row  : {'source_image': 'M19_02_1_11_Bright Field_001.jpg', 'x': 602, 'y': 986, 'polymer_score': 42, 'display_radius': 42.24038352098617, 'current_label': None, 'current_notes': ''}
polymer score: min=0 median=0 p90=0 max=1886 (n_with_score>0 = 4978)


## Launch the labeler

Press a number key to label and advance. Each keypress immediately appends a row to `labels.csv`.

In [8]:
_LEGEND_COLOR = "#1a4480"
 
_LEGEND_HIGHLIGHT = "#d33"   # red-ish; flags the cell's existing label in redo


class AnnotatorApp:
    """Tk-based labeler that operates in one of two modes:

    * ``redo_mode=False``: each keypress *appends* a new row to ``csv_path``.
      Undo truncates the file back to the byte offset taken before the append.
    * ``redo_mode=True``: each keypress *overwrites* the existing row matched
      by ``(source_image, x, y)``. Undo restores the previous label + notes.

    Hotkey contract — ``8/9/0/1/2/3/z/s`` never insert into the notes entry.
    Each handler returns ``"break"`` from a binding installed *both* on root
    AND on the entry widget itself, which is the only way Tk reliably stops the
    keystroke from also being typed into a focused ``tk.Entry``.
    """

    def __init__(
        self,
        root: tk.Tk,
        queue: list[dict],
        csv_path: Path,
        annotator: str,
        redo_mode: bool = False,
    ) -> None:
        self.root = root
        self.queue = queue
        self.csv_path = csv_path
        self.annotator = annotator
        self.redo_mode = redo_mode
        self.idx = 0
        # Each entry is either:
        #   ("append",    idx, item, byte_offset)
        #   ("overwrite", idx, item, old_label, old_notes)
        self._undo_stack: list[tuple] = []

        root.title(f"RBC Crop Annotator {'(REDO)' if redo_mode else ''}".strip())
        root.configure(padx=18, pady=14)

        self.counter_label = tk.Label(root, text="", font=("Arial", 14, "bold"))
        self.counter_label.pack(pady=(0, 6))
        self.info_label = tk.Label(root, text="", font=("Arial", 11))
        self.info_label.pack(pady=(0, 8))
        self.image_label = tk.Label(root, bg="black")
        self.image_label.pack(pady=4)

        # --- Class legend (per-key tk.Labels so we can recolor in redo) ---
        legend_frame = tk.Frame(root)
        legend_frame.pack(pady=(8, 4))
        self.legend_labels: dict[str, tk.Label] = {}
        for key, name in (
            (KEY_SICKLE, "sickle"),
            (KEY_NON_SICKLE, "non_sickle"),
            (KEY_AMBIGUOUS, "ambiguous"),
        ):
            lbl = tk.Label(
                legend_frame, text=f"{key} → {name}",
                font=("Arial", 11, "bold"), fg=_LEGEND_COLOR, padx=8,
            )
            lbl.pack(side="left")
            self.legend_labels[name] = lbl

        # Misc-action legend.
        meta_frame = tk.Frame(root)
        meta_frame.pack(pady=(0, 4))
        skip_label = "keep current" if redo_mode else "skip"
        tk.Label(
            meta_frame,
            text=f"{KEY_UNDO} → undo    {KEY_SKIP} → {skip_label}    Esc → quit",
            font=("Arial", 10), fg="#444",
        ).pack(side="left")

        # --- Notes shortcut legend (1/2/3) ---
        notes_legend_frame = tk.Frame(root)
        notes_legend_frame.pack(pady=(0, 8))
        for key, note in NOTE_SHORTCUTS.items():
            tk.Label(
                notes_legend_frame, text=f"{key} → {note}",
                font=("Arial", 10), fg="#555", padx=6,
            ).pack(side="left")

        # --- Notes entry ---
        notes_frame = tk.Frame(root)
        notes_frame.pack(pady=(4, 4), fill="x")
        tk.Label(notes_frame, text="Notes (optional):", font=("Arial", 10)).pack(anchor="w")
        self.notes_entry = tk.Entry(notes_frame, font=("Arial", 11))
        self.notes_entry.pack(fill="x", ipady=4)

        # Bind hotkeys on BOTH root and the entry. Each handler returns "break"
        # so the entry never types the digit when its binding fires.
        self._bind_hotkeys(root)
        self._bind_hotkeys(self.notes_entry)

        self.show_current()

    # -------------------------------------------------------------- bindings
    def _bind_hotkeys(self, widget: tk.Misc) -> None:
        def make(handler):
            def cb(_event):
                handler()
                return "break"
            return cb

        widget.bind(KEY_SICKLE, make(lambda: self.label_current("sickle")))
        widget.bind(KEY_NON_SICKLE, make(lambda: self.label_current("non_sickle")))
        widget.bind(KEY_AMBIGUOUS, make(lambda: self.label_current("ambiguous")))
        widget.bind(KEY_UNDO, make(self.undo))
        widget.bind(KEY_SKIP, make(self.skip))
        widget.bind("<Escape>", make(lambda: self.root.destroy()))
        for key, note in NOTE_SHORTCUTS.items():
            widget.bind(key, make(lambda n=note: self.set_note(n)))

    # -------------------------------------------------------------- rendering
    def _render(self, pil_img: Image.Image) -> ImageTk.PhotoImage:
        size = (pil_img.width * DISPLAY_SCALE, pil_img.height * DISPLAY_SCALE)
        upscaled = pil_img.resize(size, Image.NEAREST)
        return ImageTk.PhotoImage(upscaled)

    def _set_legend_highlight(self, current: str | None) -> None:
        for name, lbl in self.legend_labels.items():
            lbl.config(fg=_LEGEND_HIGHLIGHT if name == current else _LEGEND_COLOR)

    def show_current(self) -> None:
        if self.idx >= len(self.queue):
            self.counter_label.config(text="All crops processed ✓")
            self.info_label.config(text="You can close the window.")
            self.image_label.config(image="")
            self._set_legend_highlight(None)
            return
        item = self.queue[self.idx]
        self.photo = self._render(item["display"])
        self.image_label.config(image=self.photo)
        self.counter_label.config(text=f"{self.idx + 1} / {len(self.queue)}")

        parts = [
            item["source_image"],
            f"x={item['x']}",
            f"y={item['y']}",
            f"polymer_score={item['polymer_score']}",
        ]
        if self.redo_mode and item.get("current_label") is not None:
            parts.append(f"current={item['current_label']}")
        self.info_label.config(text="   •   ".join(parts))

        # Reset notes entry. In redo mode pre-populate with the prior note so
        # the user can keep / edit / clear it.
        self.notes_entry.delete(0, tk.END)
        if self.redo_mode and item.get("current_notes"):
            self.notes_entry.insert(0, item["current_notes"])

        # Highlight existing class in red (redo only).
        self._set_legend_highlight(item.get("current_label") if self.redo_mode else None)

    # ------------------------------------------------------------- CSV writes
    def _append_row(self, item: dict, new_label: str, note: str) -> int:
        offset = self.csv_path.stat().st_size if self.csv_path.exists() else 0
        with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(
                [item["source_image"], item["x"], item["y"], new_label, self.annotator, note]
            )
        return offset

    def _overwrite_row(self, item: dict, new_label: str, note: str) -> tuple[str, str]:
        df = pd.read_csv(self.csv_path)
        mask = (
            (df["source_image"] == item["source_image"])
            & (df["x"].astype(int) == int(item["x"]))
            & (df["y"].astype(int) == int(item["y"]))
        )
        if mask.sum() == 0:
            print(f"⚠ no row to overwrite for {item['source_image']} ({item['x']}, {item['y']}).")
            return ("", "")
        old_label = str(df.loc[mask, "label"].iloc[0])
        old_notes = (
            str(df.loc[mask, "notes"].fillna("").iloc[0]) if "notes" in df.columns else ""
        )
        df.loc[mask, "label"] = new_label
        df.loc[mask, "annotator"] = self.annotator
        if "notes" not in df.columns:
            df["notes"] = ""
        df.loc[mask, "notes"] = note
        df.to_csv(self.csv_path, index=False)
        item["current_label"] = new_label
        item["current_notes"] = note
        return (old_label, old_notes)

    # ------------------------------------------------------------- actions
    def set_note(self, note: str) -> None:
        self.notes_entry.delete(0, tk.END)
        self.notes_entry.insert(0, note)

    def label_current(self, label: str) -> None:
        if self.idx >= len(self.queue):
            return
        item = self.queue[self.idx]
        note = self.notes_entry.get().strip()
        if self.redo_mode:
            old_label, old_notes = self._overwrite_row(item, label, note)
            self._undo_stack.append(("overwrite", self.idx, item, old_label, old_notes))
        else:
            offset = self._append_row(item, label, note)
            self._undo_stack.append(("append", self.idx, item, offset))
        self.idx += 1
        self.show_current()

    def skip(self) -> None:
        if self.idx < len(self.queue):
            self.idx += 1
            self.show_current()

    def undo(self) -> None:
        if not self._undo_stack:
            return
        entry = self._undo_stack.pop()
        mode = entry[0]
        if mode == "append":
            _, idx, _item, offset = entry
            with open(self.csv_path, "r+b") as f:
                f.truncate(offset)
            self.idx = idx
        elif mode == "overwrite":
            _, idx, item, old_label, old_notes = entry
            if old_label:
                self._overwrite_row(item, old_label, old_notes)
            self.idx = idx
        self.show_current()


_csv_for_app = LABELS_CSV_FOR_REDO if REDO_MODE else CSV_OUTPUT_PATH
if not queue:
    print("Nothing to process. Add images to to_be_labeled/, change REDO_CLASSES,"
          " or check the CSV path.")
else:
    root = tk.Tk()
    AnnotatorApp(root, queue, _csv_for_app, ANNOTATOR_NAME, redo_mode=REDO_MODE)
    root.mainloop()
    print("Session ended.")


Session ended.


## Quick summary

In [ ]:
import pandas as pd
df = pd.read_csv(CSV_OUTPUT_PATH)
print(f"labels.csv now has {len(df)} rows.")
if not df.empty:
    print(df['label'].value_counts())
    print()
    print(df.tail(8))